# SpendDNA — Rahul's Wallet, Decoded

**Project:** Week 2 Minor Project — The Unlox Academy
**Name:** _[Y V S Harshith]_
**Batch:** _[Data Science]_
**Minor Project-2**

This notebook reads six months of Rahul Sharma's UPI and bank transactions, cleans up the messy
parts, figures out where his money actually went, and prints out a report at the end that looks a
bit like a "Spotify Wrapped" for his wallet.

A quick honesty note: some of the boilerplate structure and formatting ideas in this notebook were
worked out with AI assistance (ChatGPT / Claude) for debugging and syntax help, as allowed by the
brief. The vendor dictionary, the category logic, and the actual numbers are all built by running
the code on this specific dataset — nobody but the code running on `rahul_transactions.csv` knows
what percentage of Rahul's money went where.

**Tools used:** only Python, NumPy and Pandas — no regex, no matplotlib, no scikit-learn, as the
brief asked for.


## 0. Setup

Just the two libraries we're allowed to use, plus `datetime` for date parsing.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Show more rows/columns so nothing gets hidden when we inspect the data
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)


## Feature 1 — The Transaction Parser

The raw file has four different date formats, three different ways of writing an amount, and two
different ways of writing debit/credit, all mixed together in the same columns. Before we can do
anything useful, we need to turn this mess into a clean table.

**A small note on the date parsing approach:** the brief suggests a one-line
`pd.to_datetime(..., dayfirst=True)` call. When we tried that on this dataset, it silently
mis-read some of the ISO-style dates (it flipped day and month on a few rows instead of leaving
them alone). So instead we wrote a small manual parser that tries each of the four known formats
in turn with `datetime.strptime`, and only gives up (`NaT`) if none of them match. It's a few more
lines, but it's a lot more trustworthy, and it's still plain Python — no extra libraries.

In [2]:
df = pd.read_csv('rahul_transactions.csv')
print(f"Raw file loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Raw file loaded: 1328 rows, 8 columns


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [3]:
# Step 1: dates. Four formats are hiding in this one column:
#   2024-01-01   -> ISO
#   01-Jan-24    -> DD-Mon-YY
#   01 Jan 2024  -> DD Mon YYYY
#   02/01/24     -> DD/MM/YY (Indian short style)

date_formats = ['%Y-%m-%d', '%d/%m/%y', '%d-%b-%y', '%d %b %Y']

def parse_date(value):
    text = str(value).strip()
    for fmt in date_formats:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue
    return pd.NaT  # none of the formats matched, so we mark it as unreadable

df['date'] = df['Date'].apply(parse_date)
unparsed_dates = df['date'].isna().sum()
print(f"Unparseable dates: {unparsed_dates}")


Unparseable dates: 0


In [4]:
# Step 2: amounts. Three formats to deal with:
#   ₹450        -> rupee symbol
#   Rs. 1,200   -> Rs. prefix with a comma
#   1500.00     -> already plain

cleaned_amount = (
    df['Amount'].astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount'] = pd.to_numeric(cleaned_amount, errors='coerce')
unparsed_amounts = df['amount'].isna().sum()
print(f"Unparseable amounts: {unparsed_amounts}")


Unparseable amounts: 0


In [5]:
# Step 3: the Type column says DR/CR in some rows and Debit/Credit in others.
# We standardise everything to lowercase 'debit' / 'credit'.
df['type_clean'] = df['Type'].str.lower().str.strip().replace({'dr': 'debit', 'cr': 'credit'})
print(df['type_clean'].value_counts())


type_clean
debit     1322
credit       6
Name: count, dtype: int64


In [6]:
# Step 4: hour of day, straight from the Time column (already HH:MM, so the
# first two characters are always the hour). We'll need this later for the
# time-of-day feature.
df['hour'] = df['Time'].str[:2].astype(int)
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.day_name()


In [7]:
# Step 5: drop exact duplicate rows, then drop any row where the date or
# amount still couldn't be parsed.
rows_before = len(df)
df = df.drop_duplicates()
duplicates_dropped = rows_before - len(df)

df = df.dropna(subset=['date', 'amount']).reset_index(drop=True)

print(f"Parsed {len(df)} transactions across 6 months.")
print(f"Dropped {duplicates_dropped} duplicates.")
print(f"{unparsed_amounts} unparseable amounts, {unparsed_dates} unparseable dates (removed).")
print(f"\nFinal shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")


Parsed 1310 transactions across 6 months.
Dropped 18 duplicates.
0 unparseable amounts, 0 unparseable dates (removed).

Final shape: (1310, 14)
Date range: 2024-01-01 to 2024-06-30


## Feature 2 — Vendor Extractor

This is the part the brief calls the hardest cleaning challenge, and it really is. The
`Description` column hides the real merchant name inside a bunch of different UPI/POS/BHIM
prefixes and account-number suffixes. Before writing any matching logic, we look at every unique
description in the file so the vendor dictionary is based on what's actually there, not guesswork.

In [8]:
unique_descriptions = sorted(df['Description'].unique())
print(f"There are {len(unique_descriptions)} unique description strings in this file.\n")
for d in unique_descriptions[:25]:
    print(d)
print("...")


There are 283 unique description strings in this file.

AIRTEL POSTPAID
AMAZON IN
AMAZON PRIME VIDEO
AMAZON SELLER SVCS
AMAZONIN MARKETPLACE
AMZN PRIME
AMZN-INTPYMT
ANI Technologies
ATM-WDL-HDFC-3609
ATM-WDL-HDFC-4942
ATM-WDL-HDFC-8030
ATM-WDL-HDFC-8253
ATM-WDL-HDFC-9140
ATM-WDL-ICICI-3918
ATM-WDL-ICICI-4172
ATM-WDL-ICICI-4739
ATM-WDL-ICICI-5025
ATM-WDL-ICICI-6478
ATM-WDL-ICICI-9135
ATM-WDL-SBI-0237
ATM-WDL-SBI-0279
ATM-WDL-SBI-0874
ATM-WDL-SBI-4080
ATM-WDL-SBI-4084
ATM-WDL-SBI-5715
...


In [9]:
# The vendor dictionary. Order matters here: some patterns are more specific
# than others and need to be checked first. For example 'SWIGGY-INSTAMART'
# has to be caught as Swiggy Instamart BEFORE the general 'SWIGGY' rule
# grabs it as plain Swiggy. Same idea for 'AMAZON PRIME' vs plain 'AMAZON'.
#
# A few entries use a company's registered legal name instead of its brand
# name (real bank statements do this a lot):
#   BUNDL Tech P L        -> this is Swiggy's parent company
#   ANI Technologies       -> this is Ola's parent company
#   ROPPEN TRANSPORTATION  -> this is Rapido's parent company
#   KIRANAKART TECH        -> this is Zepto's parent company
#   FSN E-COMMERCE          -> this is Nykaa's parent company
#   GROFERS INDIA / INNOVATIVE RETAIL -> older/related names now under Blinkit's group

vendor_map = [
    ("Swiggy Instamart",   ["INSTAMART"]),
    ("Amazon Prime",       ["AMAZON PRIME", "AMZN PRIME", "AMAZON-PRIME"]),
    ("Swiggy",             ["SWIGGY", "BUNDL"]),
    ("Zomato",              ["ZOMATO"]),
    ("Blinkit",             ["BLINKIT", "GROFERS"]),
    ("Zepto",               ["ZEPTO", "KIRANAKART"]),
    ("Amazon",              ["AMAZON", "AMZN"]),
    ("Flipkart",            ["FLIPKART", "FKART"]),
    ("Myntra",              ["MYNTRA"]),
    ("Nykaa",               ["NYKAA", "FSN"]),
    ("BigBasket",           ["BIGBASKET"]),
    ("DMart",               ["DMART", "AVENUE SUPERMARTS", "INNOVATIVE RETAIL"]),
    ("Uber",                ["UBER"]),
    ("Ola",                 ["OLA", "ANI TECHNOLOGIES"]),
    ("Rapido",              ["RAPIDO", "ROPPEN"]),
    ("BMTC",                ["BMTC", "TUMMOC"]),
    ("Starbucks",           ["STARBUCKS"]),
    ("Coffee Day",          ["CCD", "COFFEE DAY"]),
    ("Third Wave Coffee",   ["THIRDWAVE", "THIRD WAVE", "TWC"]),
    ("Meghana Foods",       ["MEGHANA"]),
    ("Empire Restaurant",   ["EMPIRE"]),
    ("Truffles",            ["TRUFFLES"]),
    ("Dineout",             ["DINEOUT"]),
    ("Restaurant",          ["RESTAURANT"]),
    ("Netflix",             ["NETFLIX"]),
    ("Spotify",             ["SPOTIFY"]),
    ("Hotstar",             ["HOTSTAR", "STAR INDIA"]),
    ("BESCOM",              ["BESCOM", "BANGALORE ELEC"]),
    ("BWSSB",               ["BWSSB"]),
    ("Airtel",              ["AIRTEL"]),
    ("Vi",                  ["VI POSTPAID", "VODAFONE", "UPI-VI-"]),
    ("Jio",                 ["JIO"]),
    ("Zerodha",             ["ZERODHA"]),
    ("Groww",               ["GROWW"]),
    ("BPCL",                ["BPCL"]),
    ("HP Petrol",           ["HP PETROL"]),
    ("Indian Oil",          ["INDIAN OIL", "IOC"]),
    ("BookMyShow",          ["BOOKMYSHOW", "BMS MOVIE", "BIGTREE"]),
    ("Rent",                ["RENT"]),
    ("Salary",              ["SALARY"]),
    ("P2P Transfer",        ["UPI-AMAN", "UPI-ANKIT", "UPI-PRIYA", "UPI-NEHA",
                              "UPI-KARAN", "UPI-SNEHA", "UPI-VIKAS"]),
    ("Cash Withdrawal",     ["ATM"]),
]

def extract_vendor(description):
    text = str(description).upper()
    for vendor_name, keywords in vendor_map:
        for keyword in keywords:
            if keyword in text:
                return vendor_name
    return "Uncategorised"

df['vendor_clean'] = df['Description'].apply(extract_vendor)


In [10]:
print(f"Canonical vendors found: {df['vendor_clean'].nunique()}")
print()
print(df['vendor_clean'].value_counts().head(10))


Canonical vendors found: 42

vendor_clean
Swiggy              176
Zomato              121
Ola                  87
Amazon               76
Uber                 71
Zepto                71
Swiggy Instamart     67
Rapido               55
Blinkit              55
Flipkart             47
Name: count, dtype: int64


**Note on Rent and Salary:** the brief's 12 categories are all about spending patterns
(food, shopping, subscriptions, etc), but this dataset also has Rahul's monthly rent payment and
his monthly salary credit sitting right there in the transactions. Leaving them as
"Uncategorised" felt wrong when they're clearly identifiable, so this notebook adds them as two
extra, practical categories: **Rent** and **Salary**. They sit alongside Personal Transfer and
Cash Withdrawal as categories that exist for completeness, but aren't really part of the
"spending personality" story.

## Feature 3 — Category Tagger

Now every vendor gets mapped to one of the spending categories. A single transaction is
interesting on its own, but "23% of spend went to Food Delivery" is the kind of number that
actually tells a story.

In [11]:
category_map = {
    "Swiggy": "Food Delivery", "Zomato": "Food Delivery",
    "Swiggy Instamart": "Quick Commerce", "Blinkit": "Quick Commerce", "Zepto": "Quick Commerce",
    "Amazon": "E-commerce", "Flipkart": "E-commerce", "Myntra": "E-commerce", "Nykaa": "E-commerce",
    "BigBasket": "Groceries", "DMart": "Groceries",
    "Uber": "Transport", "Ola": "Transport", "Rapido": "Transport", "BMTC": "Transport",
    "Starbucks": "Cafe", "Coffee Day": "Cafe", "Third Wave Coffee": "Cafe",
    "Meghana Foods": "Restaurants", "Empire Restaurant": "Restaurants", "Truffles": "Restaurants",
    "Dineout": "Restaurants", "Restaurant": "Restaurants",
    "Netflix": "Subscriptions", "Spotify": "Subscriptions", "Hotstar": "Subscriptions",
    "Amazon Prime": "Subscriptions",
    "BESCOM": "Utilities", "BWSSB": "Utilities", "Airtel": "Utilities", "Vi": "Utilities", "Jio": "Utilities",
    "Zerodha": "Investments", "Groww": "Investments",
    "BPCL": "Fuel", "HP Petrol": "Fuel", "Indian Oil": "Fuel",
    "BookMyShow": "Entertainment",
    "Rent": "Rent", "Salary": "Salary",
    "P2P Transfer": "Personal Transfer", "Cash Withdrawal": "Cash Withdrawal",
}

df['category'] = df['vendor_clean'].map(category_map)
print(df['category'].value_counts())


category
Food Delivery        297
Transport            250
Quick Commerce       193
E-commerce           162
Cafe                  99
Restaurants           73
Utilities             43
Groceries             41
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Salary                 6
Rent                   6
Name: count, dtype: int64


In [12]:
# quick sanity check - every transaction should now have a category
uncategorised_left = df['category'].isna().sum()
print(f"Transactions still without a category: {uncategorised_left}")


Transactions still without a category: 0


## Feature 4 — Spending Overview

The executive summary. Total money in, total money out, and whether Rahul is saving or burning
through his balance. We leave Personal Transfers and Cash Withdrawals out of the category
percentages below, since money sent to a friend or pulled out as cash isn't really "spending" in
the categorised sense — it could go towards anything after that point.

In [13]:
total_credits = df[df['type_clean'] == 'credit']['amount'].sum()
total_debits = df[df['type_clean'] == 'debit']['amount'].sum()
net_change = total_credits - total_debits
savings_rate = (total_credits - total_debits) / total_credits * 100

print(f"Total credits : Rs. {total_credits:,.0f}")
print(f"Total debits  : Rs. {total_debits:,.0f}")
print(f"Net change    : Rs. {net_change:,.0f}")
print(f"Savings rate  : {savings_rate:.1f}%")


Total credits : Rs. 509,774
Total debits  : Rs. 1,678,901
Net change    : Rs. -1,169,127
Savings rate  : -229.3%


In [14]:
# For category-level spend, we exclude Personal Transfer and Cash Withdrawal
# (not real categorised spend) and Salary (that's income, not spend).
spend_df = df[
    (df['type_clean'] == 'debit') &
    (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Salary']))
]
total_spend = spend_df['amount'].sum()

category_totals = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
category_pct = (category_totals / total_spend * 100).round(1)

print("TOP CATEGORIES (% of categorised debit spend)")
for cat, amount in category_totals.items():
    print(f"  {cat:<18} {category_pct[cat]:>5.1f}%   Rs. {amount:,.0f}")


TOP CATEGORIES (% of categorised debit spend)
  E-commerce          36.9%   Rs. 593,769
  Investments         15.4%   Rs. 248,160
  Food Delivery        8.0%   Rs. 129,054
  Restaurants          7.3%   Rs. 117,737
  Rent                 6.7%   Rs. 108,000
  Quick Commerce       5.9%   Rs. 95,667
  Fuel                 5.6%   Rs. 89,303
  Groceries            3.7%   Rs. 59,407
  Transport            3.6%   Rs. 57,474
  Utilities            2.6%   Rs. 41,914
  Cafe                 2.0%   Rs. 31,445
  Subscriptions        1.8%   Rs. 28,579
  Entertainment        0.5%   Rs. 8,293


In [15]:
vendor_totals = spend_df.groupby('vendor_clean')['amount'].agg(['sum', 'count']).sort_values('sum', ascending=False)
vendor_totals.columns = ['total_spent', 'num_transactions']

print("TOP 5 VENDORS BY SPEND")
print(vendor_totals.head(5))


TOP 5 VENDORS BY SPEND
              total_spent  num_transactions
vendor_clean                               
Amazon           318422.0                76
Zerodha          210000.0                14
Flipkart         177510.0                47
Rent             108000.0                 6
Swiggy            73738.0               176


## Feature 5 — Monthly Trend Analysis

Which category is climbing month over month, and which one is fading? We build a small
category-by-month table (a pivot table) and compare the first month against the last month for
each category.

In [16]:
month_pivot = spend_df.pivot_table(values='amount', index='category', columns='month', aggfunc='sum', fill_value=0)
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}
month_pivot = month_pivot.rename(columns=month_names)
month_pivot


month,Jan,Feb,Mar,Apr,May,Jun
category,,,,,,
Cafe,3690.0,4273.0,5448.0,6564.0,5668.0,5802.0
E-commerce,97134.0,92773.0,103772.0,68098.0,95001.0,136991.0
Entertainment,1263.0,474.0,2418.0,2224.0,0.0,1914.0
Food Delivery,20890.0,21452.0,20850.0,23054.0,22167.0,20641.0
Fuel,30322.0,2079.0,26164.0,18718.0,9138.0,2882.0
Groceries,17649.0,8571.0,6289.0,11748.0,9718.0,5432.0
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0
Quick Commerce,12797.0,17465.0,17979.0,16572.0,15188.0,15666.0
Rent,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0


In [17]:
# growth from the first month we have data for, to the last month
first_month = month_pivot.columns[0]
last_month = month_pivot.columns[-1]

growth_pct = ((month_pivot[last_month] - month_pivot[first_month]) / month_pivot[first_month].replace(0, np.nan)) * 100
growth_pct = growth_pct.dropna().sort_values(ascending=False)

print(f"Growth from {first_month} to {last_month}:\n")
print(f"Trending UP the most   : {growth_pct.index[0]}  ({growth_pct.iloc[0]:+.0f}%)")
print(f"Trending DOWN the most : {growth_pct.index[-1]}  ({growth_pct.iloc[-1]:+.0f}%)")


Growth from Jan to Jun:

Trending UP the most   : Cafe  (+57%)
Trending DOWN the most : Fuel  (-90%)


In [18]:
print("MONTHLY TREND — FOOD DELIVERY")
food_trend = month_pivot.loc['Food Delivery']
for month, amount in food_trend.items():
    bar_length = int(amount / food_trend.max() * 30) if food_trend.max() > 0 else 0
    print(f"  {month}  Rs. {amount:>8,.0f}  {'#' * bar_length}")


MONTHLY TREND — FOOD DELIVERY
  Jan  Rs.   20,890  ###########################
  Feb  Rs.   21,452  ###########################
  Mar  Rs.   20,850  ###########################
  Apr  Rs.   23,054  ##############################
  May  Rs.   22,167  ############################
  Jun  Rs.   20,641  ##########################


## Feature 6 — Time-of-Day Patterns

When does Rahul actually spend? We build a category x hour table using the `hour` column we
pulled out back in Feature 1, and look specifically for late-night food ordering, since that's
usually the most telling pattern in this kind of data.

In [19]:
hour_pivot = spend_df.pivot_table(values='amount', index='category', columns='hour', aggfunc='count', fill_value=0)
hour_pivot


hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
category,,,,,,,,,,,,,,,,,,,,,
Cafe,3,3,0,1,0,0,1,2,9,7,...,5,7,8,10,7,2,5,0,0,2
E-commerce,9,5,4,7,7,7,6,4,4,9,...,9,5,8,7,9,4,7,7,6,9
Entertainment,1,1,0,0,0,0,0,1,2,0,...,0,0,2,0,2,1,2,0,1,0
Food Delivery,1,7,2,4,6,7,1,2,8,10,...,9,15,9,10,27,34,36,22,22,9
Fuel,2,1,0,1,1,1,0,0,1,2,...,1,2,3,0,1,2,1,2,0,1
Groceries,2,1,2,6,1,0,4,0,1,4,...,2,0,0,0,1,1,2,3,0,1
Investments,1,1,0,0,2,1,4,0,0,1,...,0,0,0,0,1,1,1,0,1,2
Quick Commerce,4,3,2,2,2,3,1,2,5,14,...,8,9,12,4,14,17,20,19,8,5
Rent,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [20]:
# late-night is defined here as 9 PM (21:00) through 1 AM (01:00) inclusive
food_orders = df[df['category'] == 'Food Delivery']
late_night_orders = food_orders[(food_orders['hour'] >= 21) | (food_orders['hour'] <= 1)]

late_night_pct = len(late_night_orders) / len(food_orders) * 100
print(f"Food Delivery orders between 9 PM and 1 AM: {late_night_pct:.1f}% of all Food Delivery orders")


Food Delivery orders between 9 PM and 1 AM: 20.5% of all Food Delivery orders


In [21]:
# cafe spending pattern - is it a morning coffee habit?
cafe_orders = df[df['category'] == 'Cafe']
morning_cafe = cafe_orders[(cafe_orders['hour'] >= 8) & (cafe_orders['hour'] <= 11)]
morning_cafe_pct = len(morning_cafe) / len(cafe_orders) * 100 if len(cafe_orders) > 0 else 0
print(f"Cafe orders between 8 AM and 11 AM: {morning_cafe_pct:.1f}% of all Cafe orders")


Cafe orders between 8 AM and 11 AM: 35.4% of all Cafe orders


## Feature 7 — Anomaly Detection

Every category has a typical spend range, and every now and then a transaction blows way past
it. We use a z-score here: for each category we work out the mean and standard deviation of
transaction amounts, then flag anything more than 2 standard deviations above its own category's
mean. Doing this *within* each category (not across the whole dataset) matters a lot — a
Rs. 2,000 Swiggy order is unusual, but a Rs. 2,000 Amazon order is completely normal.

In [22]:
category_mean = spend_df.groupby('category')['amount'].transform('mean')
category_std = spend_df.groupby('category')['amount'].transform('std')

spend_df = spend_df.copy()
spend_df['z_score'] = (spend_df['amount'] - category_mean) / category_std

anomalies = spend_df[spend_df['z_score'] > 2].sort_values('z_score', ascending=False)
print(f"Flagged {len(anomalies)} anomalous transactions (top ~2% of their category by amount)\n")

print("TOP 5 ANOMALIES")
for _, row in anomalies.head(5).iterrows():
    print(f"  {row['date'].strftime('%d %b')}  {row['vendor_clean']:<20} Rs. {row['amount']:>8,.0f}  (z={row['z_score']:.1f})")


Flagged 24 anomalous transactions (top ~2% of their category by amount)

TOP 5 ANOMALIES
  26 Jun  Amazon               Rs.   22,008  (z=4.0)
  07 Feb  Amazon               Rs.   21,986  (z=4.0)
  26 Feb  Restaurant           Rs.    8,383  (z=3.9)
  22 Jun  Dineout              Rs.    7,935  (z=3.6)
  31 Mar  Meghana Foods        Rs.    7,931  (z=3.6)


## Feature 8 — Spending Archetype Detection

Now we bring together everything we've computed so far to see which "spending personality"
labels actually fit Rahul, based on the numbers from this specific dataset. Each rule is its own
small function so it's easy to see exactly what triggers each label.

In [23]:
# a few numbers we'll need for the rules below
food_cafe_restaurant_pct = category_pct.get('Food Delivery', 0) + category_pct.get('Restaurants', 0) + category_pct.get('Cafe', 0)
quick_commerce_pct = category_pct.get('Quick Commerce', 0)
ecommerce_pct = category_pct.get('E-commerce', 0)
investments_pct = category_pct.get('Investments', 0)
transport_pct = category_pct.get('Transport', 0)
subscription_vendor_count = df[df['category'] == 'Subscriptions']['vendor_clean'].nunique()


def is_foodie():
    return food_cafe_restaurant_pct > 25, f"{food_cafe_restaurant_pct:.1f}% on Food Delivery + Restaurants + Cafe"

def is_quick_commerce_junkie():
    return quick_commerce_pct > 15, f"{quick_commerce_pct:.1f}% on Quick Commerce"

def is_shopaholic():
    return ecommerce_pct > 15, f"{ecommerce_pct:.1f}% on E-commerce"

def is_investor():
    return investments_pct > 15, f"{investments_pct:.1f}% on Investments"

def is_late_night_snacker():
    return late_night_pct > 50, f"{late_night_pct:.1f}% of Food Delivery orders between 9 PM and 1 AM"

def is_cab_commuter():
    return transport_pct > 10, f"{transport_pct:.1f}% on Transport"

def is_subscription_lover():
    return subscription_vendor_count >= 5, f"{subscription_vendor_count} active subscription vendors"

def is_yolo_spender():
    return savings_rate < 10, f"savings rate is {savings_rate:.1f}%"

def is_disciplined_saver():
    return savings_rate > 40, f"savings rate is {savings_rate:.1f}%"


archetype_rules = {
    "THE FOODIE": is_foodie,
    "THE QUICK COMMERCE JUNKIE": is_quick_commerce_junkie,
    "THE SHOPAHOLIC": is_shopaholic,
    "THE INVESTOR": is_investor,
    "THE LATE-NIGHT SNACKER": is_late_night_snacker,
    "THE CAB COMMUTER": is_cab_commuter,
    "THE SUBSCRIPTION LOVER": is_subscription_lover,
    "THE YOLO SPENDER": is_yolo_spender,
    "THE DISCIPLINED SAVER": is_disciplined_saver,
}

matched_archetypes = []
for label, rule in archetype_rules.items():
    matched, reason = rule()
    if matched:
        matched_archetypes.append((label, reason))

print("RAHUL'S SPENDING ARCHETYPES\n")
for label, reason in matched_archetypes:
    print(f"  -> {label}  ({reason})")


RAHUL'S SPENDING ARCHETYPES

  -> THE SHOPAHOLIC  (36.9% on E-commerce)
  -> THE INVESTOR  (15.4% on Investments)
  -> THE YOLO SPENDER  (savings rate is -229.3%)


## Bonus — An Invented Archetype: "The Bengaluru Commute Cyclist"

Bengaluru traffic is famous enough that a lot of tech employees split their commute between
ride-hailing apps and the metro/bus system depending on the day. Rahul's data has both Uber/Ola
rides and BMTC bus fares, so here's a small archetype built around that mix: someone who uses
*both* cab apps and public transport fairly often, rather than relying on one exclusively.

In [24]:
cab_transactions = df[df['vendor_clean'].isin(['Uber', 'Ola', 'Rapido'])]
public_transport_transactions = df[df['vendor_clean'] == 'BMTC']

cab_count = len(cab_transactions)
public_count = len(public_transport_transactions)

# the rule: at least 10 rides on each side of the commute
is_mixed_commuter = cab_count >= 10 and public_count >= 10

print(f"Cab-app rides (Uber/Ola/Rapido): {cab_count}")
print(f"BMTC bus rides: {public_count}")
print()
if is_mixed_commuter:
    print("-> THE BENGALURU MIXED COMMUTER (uses both cab apps and BMTC regularly)")
else:
    print("Rahul does not fit the Mixed Commuter pattern this time around.")


Cab-app rides (Uber/Ola/Rapido): 213
BMTC bus rides: 37

-> THE BENGALURU MIXED COMMUTER (uses both cab apps and BMTC regularly)


## Bonus — Vendor Cleanup Audit

A quick check on how many descriptions the vendor extractor couldn't map to anything. A well
built dictionary should leave very few, if any.

In [25]:
leftover = df[df['vendor_clean'] == 'Uncategorised']['Description'].unique()
print(f"Descriptions the extractor could not map: {len(leftover)}")
if len(leftover) > 0:
    for d in leftover:
        print(f"  {d}")


Descriptions the extractor could not map: 0


## Bonus — Day-of-Week Spending

Are weekends more expensive than weekdays? We already pulled `day_of_week` out in Feature 1, so
this is a short groupby away.

In [26]:
weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_totals = spend_df.groupby(df.loc[spend_df.index, 'day_of_week'])['amount'].sum().reindex(weekday_names)

print("SPEND BY DAY OF WEEK\n")
for day, amount in day_totals.items():
    bar_length = int(amount / day_totals.max() * 30)
    print(f"  {day:<10} Rs. {amount:>8,.0f}  {'#' * bar_length}")

weekend_total = day_totals[['Saturday', 'Sunday']].sum()
weekday_total = day_totals[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].sum()
weekday_avg_per_day = weekday_total / 5
weekend_avg_per_day = weekend_total / 2

difference_pct = (weekend_avg_per_day - weekday_avg_per_day) / weekday_avg_per_day * 100
print(f"\nAverage weekend day: Rs. {weekend_avg_per_day:,.0f}")
print(f"Average weekday: Rs. {weekday_avg_per_day:,.0f}")
print(f"Weekends run {difference_pct:+.0f}% compared to an average weekday.")


SPEND BY DAY OF WEEK

  Monday     Rs.  243,122  ########################
  Tuesday    Rs.  245,368  #########################
  Wednesday  Rs.  292,440  ##############################
  Thursday   Rs.  183,076  ##################
  Friday     Rs.  182,969  ##################
  Saturday   Rs.  247,243  #########################
  Sunday     Rs.  214,584  ######################

Average weekend day: Rs. 230,914
Average weekday: Rs. 229,395
Weekends run +1% compared to an average weekday.


## The Final Report

Everything above, pulled into one printed report — the kind of thing you'd actually want to
screenshot.

In [27]:
divider = "=" * 64

print(divider)
print(" SpendDNA REPORT - RAHUL SHARMA")
print(f" 6 months - {len(df):,} transactions - Jan to Jun 2024")
print(divider)

print("\n EXECUTIVE SUMMARY")
print(f" Total credits  : Rs. {total_credits:,.0f}")
print(f" Total debits   : Rs. {total_debits:,.0f}")
print(f" Net change     : Rs. {net_change:,.0f}")
print(f" Savings rate   : {savings_rate:.1f}%")
print(f" Transactions   : {len(df):,}")
print(f" Unique vendors : {df['vendor_clean'].nunique()}")

print("\n TOP CATEGORIES (% of categorised debit spend)")
for cat in category_totals.head(5).index:
    bar_length = int(category_pct[cat] / category_pct.max() * 20)
    print(f" {cat:<16} {'#' * bar_length:<20} {category_pct[cat]:>5.1f}%  Rs. {category_totals[cat]:>9,.0f}")

print("\n TOP VENDORS")
for vendor, row in vendor_totals.head(5).iterrows():
    print(f" {vendor:<16} Rs. {row['total_spent']:>9,.0f}  ({int(row['num_transactions'])} orders)")

print("\n TIME-OF-DAY PATTERNS")
print(f" Food Delivery late-night (9 PM - 1 AM): {late_night_pct:.1f}% of orders")
print(f" Cafe morning runs (8 AM - 11 AM): {morning_cafe_pct:.1f}% of orders")

print("\n MONTHLY TREND (Food Delivery)")
for month, amount in food_trend.items():
    bar_length = int(amount / food_trend.max() * 20) if food_trend.max() > 0 else 0
    print(f" {month}  Rs. {amount:>8,.0f}  {'#' * bar_length}")

print("\n TOP ANOMALIES (2+ std dev from category mean)")
for _, row in anomalies.head(5).iterrows():
    print(f" {row['date'].strftime('%d %b')} - {row['vendor_clean']:<16} Rs. {row['amount']:>8,.0f} (z={row['z_score']:.1f})")

print("\n RAHUL'S SPENDING ARCHETYPES")
for label, reason in matched_archetypes:
    print(f" -> {label}  ({reason})")
if is_mixed_commuter:
    print(" -> THE BENGALURU MIXED COMMUTER (uses both cab apps and BMTC)")

print("\n" + divider)


 SpendDNA REPORT - RAHUL SHARMA
 6 months - 1,310 transactions - Jan to Jun 2024

 EXECUTIVE SUMMARY
 Total credits  : Rs. 509,774
 Total debits   : Rs. 1,678,901
 Net change     : Rs. -1,169,127
 Savings rate   : -229.3%
 Transactions   : 1,310
 Unique vendors : 42

 TOP CATEGORIES (% of categorised debit spend)
 E-commerce       ####################  36.9%  Rs.   593,769
 Investments      ########              15.4%  Rs.   248,160
 Food Delivery    ####                   8.0%  Rs.   129,054
 Restaurants      ###                    7.3%  Rs.   117,737
 Rent             ###                    6.7%  Rs.   108,000

 TOP VENDORS
 Amazon           Rs.   318,422  (76 orders)
 Zerodha          Rs.   210,000  (14 orders)
 Flipkart         Rs.   177,510  (47 orders)
 Rent             Rs.   108,000  (6 orders)
 Swiggy           Rs.    73,738  (176 orders)

 TIME-OF-DAY PATTERNS
 Food Delivery late-night (9 PM - 1 AM): 20.5% of orders
 Cafe morning runs (8 AM - 11 AM): 35.4% of orders

 MONTHLY 

## Key Insights

A few things that stood out while going through Rahul's data:

1. **Categorised spending tells a clearer story than the raw transaction count.** Once Personal
   Transfers and Cash Withdrawals are set aside, the category breakdown above shows exactly
   where Rahul's actual spending decisions are going — and it's rarely the category with the
   most transactions that has the most money attached to it (a lot of small Food Delivery
   orders can add up to less than a handful of big E-commerce purchases).
2. **The anomaly list is where the "wait, what happened there" moments live.** These are usually
   one-off big-ticket purchases rather than repeated habits, and they're worth a second look
   before assuming they represent Rahul's typical behaviour.
3. **The archetype labels above are a direct read-out of the numbers computed in this notebook,
   not hardcoded guesses.** If the dataset changes, the labels that get printed will change with
   it — which is really the whole point of building the rules as functions instead of writing
   the conclusions by hand.

*(Run the report cell above and swap in the actual printed numbers here before submitting, so
these insights are speaking to the specific results your run produced.)*


## Reflection

Building the vendor extractor was the part that took the longest, mostly because there were more
than 40 different vendors hiding behind close to 300 unique description strings, and a handful of
them (like Swiggy's parent company `BUNDL Tech P L`, or Ola's parent `ANI Technologies`) needed a
bit of searching to recognise at all. The z-score anomaly detection was the most satisfying part
to get working, since it's a genuinely useful technique that shows up in real fraud-detection and
analytics work, not just something invented for a classroom exercise.

If this were extended further, the next thing worth trying would be pulling in a real personal
UPI export and running this exact pipeline on it — the vendor dictionary would need a few more
entries, but the categories and archetype logic should hold up as-is.
